# 2. Data Wrangling — Cleaning, Web Scraping & Labeling

This notebook resolves the API's raw ids into readable columns, supplements the
record with details scraped from the Falcon 9 Wikipedia page, handles missing
values, and derives the binary landing-outcome label used for modeling.

## 2.1 Web scraping supplementary launch detail

In [ ]:
import requests
from bs4 import BeautifulSoup

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches"
response = requests.get(static_url)
soup = BeautifulSoup(response.text, 'html.parser')

print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


In [ ]:
# Extract all wikitable elements — one holds the launch-by-launch record
tables = soup.find_all('table', "wikitable plainrowheaders collapsible")
print(f"Found {len(tables)} candidate tables")
column_names = [th.get_text(strip=True) for th in tables[2].find_all('th')]
print(column_names[:8])

## 2.2 Resolving ids from the API into readable fields

In [ ]:
import pandas as pd

# (Continuing from 01_data_collection_api.ipynb's `falcon9_only` DataFrame)
# Look up launchpad name, payload mass/orbit, and core landing outcome per launch.
# Each of these hits a lightweight SpaceX API endpoint keyed by the id already
# present in falcon9_only, so we cache each id -> value the first time it's seen
# to avoid re-fetching the same launchpad/core/payload repeatedly.

launchpad_cache = {}
def get_launchpad_name(lp_id):
    if lp_id not in launchpad_cache:
        r = requests.get(f"https://api.spacexdata.com/v4/launchpads/{lp_id}").json()
        launchpad_cache[lp_id] = r['name']
    return launchpad_cache[lp_id]

payload_cache = {}
def get_payload_info(payload_id):
    if payload_id not in payload_cache:
        r = requests.get(f"https://api.spacexdata.com/v4/payloads/{payload_id}").json()
        payload_cache[payload_id] = (r.get('mass_kg'), r.get('orbit'))
    return payload_cache[payload_id]

core_cache = {}
def get_core_info(core):
    """core here is the per-launch 'cores' dict already embedded in falcon9_only
    (contains a core id plus the landing fields for that specific flight)."""
    core_id = core.get('core')
    if core_id not in core_cache:
        r = requests.get(f"https://api.spacexdata.com/v4/cores/{core_id}").json()
        core_cache[core_id] = r.get('serial'), r.get('reuse_count', 0), r.get('block')
    serial, reuse_count, block = core_cache[core_id]
    return {
        'Serial': serial,
        'ReusedCount': reuse_count,
        'Block': block,
        'GridFins': core.get('gridfins'),
        'Reused': core.get('reused'),
        'Legs': core.get('legs'),
        'LandingPad': core.get('landpad'),
        'LandingSuccess': core.get('landing_success'),
        'LandingType': core.get('landing_type'),
    }

# Apply each lookup across every row of the falcon9_only DataFrame.
falcon9_only['LaunchSite'] = falcon9_only['launchpad'].apply(get_launchpad_name)

payload_info = falcon9_only['payloads'].apply(get_payload_info)
falcon9_only['PayloadMass'] = payload_info.apply(lambda t: t[0])
falcon9_only['Orbit'] = payload_info.apply(lambda t: t[1])

core_info = falcon9_only['cores'].apply(get_core_info)
core_info_df = pd.DataFrame(list(core_info))
falcon9_only = pd.concat([falcon9_only.reset_index(drop=True), core_info_df], axis=1)

print(falcon9_only[['FlightNumber', 'LaunchSite', 'PayloadMass', 'Orbit', 'Serial']].head())

   FlightNumber    LaunchSite  PayloadMass Orbit   Serial
0             1  CCAFS SLC 40       6104.96   LEO   B0003
1             2  CCAFS SLC 40        525.00   LEO   B0005
2             3  CCAFS SLC 40        677.00   ISS   B0007
3             4   VAFB SLC 4E        500.00    PO   B1003
4             5  CCAFS SLC 40       3170.00   GTO   B1004


## 2.3 Deriving the landing outcome label

In [ ]:
def landing_class(core):
    """Return 1 if the first-stage core successfully landed, else 0."""
    landing_success = core.get('landing_success')
    if landing_success is True:
        return 1
    return 0

df = pd.read_csv('../data/spacex_launch_data.csv')
print(df.shape)
df.head()

(90, 17)


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

FlightNumber      0
Date              0
BoosterVersion    0
PayloadMass       0
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad       26
Block             0
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64


In [ ]:
# LandingPad is legitimately missing for flights that didn't attempt a droneship/
# ground-pad landing (e.g. expendable or ocean-landing missions) — we leave these
# as NaN rather than imputing, since "no landing pad" is itself informative.

print("Rows with missing LandingPad:", df['LandingPad'].isnull().sum())
print(df.loc[df['LandingPad'].isnull(), 'Outcome'].value_counts())

Rows with missing LandingPad: 26
None None      19
False Ocean     2
None ASDS       2
True Ocean      1... (etc.)


In [ ]:
df.to_csv('../data/spacex_launch_data.csv', index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (90, 17)
